In [1]:
# Import necessary packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fuzzywuzzy import process

c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [5]:
file_path = '/Users/brake/OneDrive/Documenten/GitHub/Thesis/Thesis/MSc-Thesis-main-NS_Train - copy/definitive_data/ns_data.csv'
def_df = pd.read_csv(file_path)

In [6]:
# DATA TRANSFORMATION 1
# Optimize data types to reduce memory usage
# ns_data['Service:RDT-ID'] = ns_data['Service:RDT-ID'].astype('int32')
# ns_data['Service:Train number'] = ns_data['Service:Train number'].astype('int32')
# ns_data['Service:Maximum delay'] = ns_data['Service:Maximum delay'].astype('int16')
# ns_data['Stop:RDT-ID'] = ns_data['Stop:RDT-ID'].astype('int32')
# ns_data['Stop:Arrival delay'] = ns_data['Stop:Arrival delay'].astype('float32')
# ns_data['Stop:Departure delay'] = ns_data['Stop:Departure delay'].astype('float32')

# Process data in chunks
chunk_size = 1000000
chunks = []

for start in range(0, len(def_df), chunk_size):
    chunk = def_df.iloc[start:start + chunk_size]
    grouped = chunk.groupby('Service:RDT-ID')

    rdt_ids = grouped['Service:RDT-ID'].unique()
    trajectories = grouped['Stop:Station name'].agg(['first', 'last'])
    trajectories = trajectories.agg(' - '.join, axis=1)
    dates = grouped['Service:Date'].first()
    day_of_week = pd.to_datetime(dates).dt.day_name()
    max_delays = grouped['Service:Maximum delay'].first()
    arrival_delays_last_stop = grouped['Stop:Arrival delay'].last()
    planned_stops = np.maximum(grouped.size() - 2, 0)  # Subtract 2 for departure and arrival stops
    cancelled_arrivals = grouped['Stop:Arrival cancelled'].sum()
    cancelled_departures = grouped['Stop:Departure cancelled'].sum()
    delayed_arrivals = (chunk['Stop:Arrival delay'] > 0).groupby(chunk['Service:RDT-ID']).sum()
    delayed_departures = (chunk['Stop:Departure delay'] > 0).groupby(chunk['Service:RDT-ID']).sum()
    partly_cancelled = grouped['Service:Partly cancelled'].any()
    completely_cancelled = grouped['Service:Completely cancelled'].all()
    last_stop_cancelled = grouped.last()['Stop:Arrival cancelled']

    chunk_df = pd.DataFrame({
        'RDT-ID': rdt_ids,
        'Trajectory': trajectories,
        'Date': dates,
        'Day of the Week': day_of_week,
        'Maximum Delay': max_delays,
        'Arrival Delay of Last Stop (min)': arrival_delays_last_stop,
        'Nr. of Planned Stops': planned_stops,
        'Nr. of Cancelled Arrivals': cancelled_arrivals,
        'Nr. of Cancelled Departures': cancelled_departures,
        'Nr. of Delayed Arrivals': delayed_arrivals,
        'Nr. of Delayed Departures': delayed_departures,
        'Partly Cancelled': partly_cancelled,
        'Completely Cancelled': completely_cancelled,
        'Last Arrival Cancelled': last_stop_cancelled
    })

    # Handling of the NaN values in "Arrival Delay of Last Stop (min)" and None values in "Last Arrival Cancelled"
    chunk_df['Arrival Delay of Last Stop (min)'].fillna(-1, inplace=True)
    chunk_df['Last Arrival Cancelled'].fillna(True, inplace=True)

    chunks.append(chunk_df)

# Concatenate all chunks into the final DataFrame
definite_df = pd.concat(chunks, ignore_index=True)

C:\Users\brake\AppData\Local\Temp\ipykernel_25404\3376959946.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  chunk_df['Arrival Delay of Last Stop (min)'].fillna(-1, inplace=True)
C:\Users\brake\AppData\Local\Temp\ipykernel_25404\3376959946.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [7]:
definite_df.to_csv('/Users/brake/OneDrive/Documenten/GitHub/Thesis/Thesis/MSc-Thesis-main-NS_Train - copy/definitive_data/definite_df.csv', index=False)

In [8]:
definite_df.head(10)

,RDT-ID,Trajectory,Date,Day of the Week,Maximum Delay,Arrival Delay of Last Stop (min),Nr. of Planned Stops,Nr. of Cancelled Arrivals,Nr. of Cancelled Departures,Nr. of Delayed Arrivals,Nr. of Delayed Departures,Partly Cancelled,Completely Cancelled,Last Arrival Cancelled
0,[738804],Rotterdam Centraal - Utrecht Centraal,2019-01-01,Tuesday,1,0.0,5,0,0,2,2,False,False,False
1,[738805],Utrecht Centraal - Rotterdam Centraal,2019-01-01,Tuesday,2,0.0,6,0,0,2,1,False,False,False
2,[738806],Rotterdam Centraal - Utrecht Centraal,2019-01-01,Tuesday,2,0.0,5,0,0,2,2,False,False,False
3,[738807],Utrecht Centraal - Rotterdam Centraal,2019-01-01,Tuesday,2,2.0,5,0,0,2,0,False,False,False
4,[738808],Rotterdam Centraal - Utrecht Centraal,2019-01-01,Tuesday,1,0.0,5,0,0,0,2,False,False,False
5,[738809],Utrecht Centraal - Rotterdam Centraal,2019-01-01,Tuesday,1,0.0,5,0,0,1,1,False,False,False
6,[738810],Lelystad Centrum - Hoofddorp,2019-01-01,Tuesday,4,4.0,12,0,0,12,13,False,False,False
7,[738811],Amersfoort - Amsterdam Centraal,2019-01-01,Tuesday,4,1.0,9,0,0,9,10,False,False,False
8,[738812],Rotterdam Centraal - Utrecht Centraal,2019-01-01,Tuesday,0,0.0,5,0,0,0,0,False,False,False
9,[738813],Alkmaar - Amsterdam Centraal,2019-01-01,Tuesday,0,0.0,9,0,0,0,0,False,False,False


In [9]:
# DATA TRANSFORMATION 2
definite_df['Date'] = pd.to_datetime(definite_df['Date'])
definite_df[['source', 'target']] = definite_df['Trajectory'].str.split(' - ', expand=True)

definite_df['source'] = definite_df['source'].replace({'Amersfoort': 'Amersfoort Centraal', 'Eindhoven': 'Eindhoven Centraal'})
definite_df['target'] = definite_df['target'].replace({'Amersfoort': 'Amersfoort Centraal', 'Eindhoven': 'Eindhoven Centraal'})

rides_performed = definite_df.dropna(subset=['Arrival Delay of Last Stop (min)']).groupby(['Date','source', 'target']).size().reset_index(name='Rides planned')
delayed_arrivals = definite_df[definite_df['Arrival Delay of Last Stop (min)'] > 0.0].groupby(['Date', 'source', 'target']).size().reset_index(name='Final arrival delay')
arrival_canceled = definite_df.groupby(['Date', 'source', 'target'])['Last Arrival Cancelled'].sum().reset_index(name='Final arrival cancelled')
trajectory_canceled = definite_df.groupby(['Date', 'source', 'target'])['Completely Cancelled'].sum().reset_index(name='Completely cancelled')
intermediate_delays = definite_df[definite_df['Nr. of Delayed Arrivals'] > 0].groupby(['Date', 'source', 'target']).size().reset_index(name='Intermediate arrival delays')

trajectories_per_day = rides_performed.merge(delayed_arrivals, on=['Date', 'source', 'target'], how='left')
trajectories_per_day = trajectories_per_day.merge(arrival_canceled, on=['Date', 'source', 'target'], how='left')
trajectories_per_day = trajectories_per_day.merge(trajectory_canceled, on=['Date', 'source', 'target'], how='left')
trajectories_per_day = trajectories_per_day.merge(intermediate_delays, on=['Date', 'source', 'target'], how='left')

trajectories_per_day['Final arrival delay'].fillna(0, inplace=True)
trajectories_per_day['Final arrival delay'] = trajectories_per_day['Final arrival delay'].astype(int)
trajectories_per_day['Final arrival cancelled'] = trajectories_per_day['Final arrival cancelled'].replace(False, 0).astype(int)
trajectories_per_day['Intermediate arrival delays'] = trajectories_per_day['Intermediate arrival delays'].fillna(0).astype(int)

# Remove rows where source is the same as target
trajectories_per_day = trajectories_per_day[trajectories_per_day['source'] != trajectories_per_day['target']]

C:\Users\brake\AppData\Local\Temp\ipykernel_25404\3437742489.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  trajectories_per_day['Final arrival delay'].fillna(0, inplace=True)


In [12]:
trajectories_per_day

,Date,source,target,Rides planned,Final arrival delay,Final arrival cancelled,Completely cancelled,Intermediate arrival delays
0,2019-01-01,'s-Hertogenbosch,Arnhem Centraal,1,0,0,0,1
1,2019-01-01,'s-Hertogenbosch,Den Haag Centraal,33,4,2,0,27
2,2019-01-01,'s-Hertogenbosch,Deurne,17,4,1,0,11
3,2019-01-01,'s-Hertogenbosch,Dordrecht,11,0,0,0,3
4,2019-01-01,'s-Hertogenbosch,Eindhoven Centraal,18,2,0,0,7
...,...,...,...,...,...,...,...,...
655782,2024-12-31,Zwolle,Nijmegen,1,0,0,0,1
655783,2024-12-31,Zwolle,Roosendaal,31,5,6,0,28
655784,2024-12-31,Zwolle,Schiphol Airport,1,0,0,0,0
655785,2024-12-31,Zwolle,Utrecht Centraal,29,1,0,0,19


In [11]:
trajectories_per_day.to_csv('/Users/brake/OneDrive/Documenten/GitHub/Thesis/Thesis/MSc-Thesis-main-NS_Train - copy/definitive_data/trajectories_per_day.csv', index=False)

In [13]:
# DATA TRANSFORMATION 3
# Convert the Date column to datetime format and create a YearMonth column
trajectories_per_day['Date'] = pd.to_datetime(trajectories_per_day['Date'])
trajectories_per_day['YearMonth'] = trajectories_per_day['Date'].dt.to_period('M')

# Perform monthly aggregation
monthly_trajectories = trajectories_per_day.groupby(['YearMonth', 'source', 'target'], as_index=False).agg({
    'Rides planned': 'sum',
    'Final arrival delay': 'sum',
    'Final arrival cancelled': 'sum',
    'Completely cancelled': 'sum',
    'Intermediate arrival delays': 'sum'
})

# Filter out trajectories with fewer than 4 rides planned per month
min_rides_threshold = 4
print(len(monthly_trajectories))
monthly_trajectories = monthly_trajectories[monthly_trajectories['Rides planned'] >= min_rides_threshold]
print(len(monthly_trajectories))

# Recalculate the proportion delayed after monthly aggregation
monthly_trajectories['Proportion delayed'] = monthly_trajectories.apply(
    lambda row: 0 if row['Final arrival delay'] == 0 else 
    row['Final arrival delay'] / (row['Rides planned'] - row['Completely cancelled']) 
    if (row['Rides planned'] - row['Completely cancelled']) > 0 else 0, axis=1)

# Calculate the 90th percentile threshold
percentile_50 = monthly_trajectories['Proportion delayed'].quantile(0.50)

# Add a column to indicate if the delay is significant based on the threshold
monthly_trajectories['Significant Delay'] = monthly_trajectories['Proportion delayed'] > percentile_50

51877
39928


In [14]:
monthly_trajectories.to_csv('/Users/brake/OneDrive/Documenten/GitHub/Thesis/Thesis/MSc-Thesis-main-NS_Train - copy/definitive_data/monthly_trajectories.csv', index=False)

In [15]:
monthly_trajectories

,YearMonth,source,target,Rides planned,Final arrival delay,Final arrival cancelled,Completely cancelled,Intermediate arrival delays,Proportion delayed,Significant Delay
1,2019-01,'s-Hertogenbosch,Arnhem Centraal,57,20,4,0,36,0.350877,True
2,2019-01,'s-Hertogenbosch,Den Haag Centraal,1106,99,20,5,853,0.089918,False
3,2019-01,'s-Hertogenbosch,Deurne,757,247,8,2,503,0.327152,True
4,2019-01,'s-Hertogenbosch,Dordrecht,100,23,3,0,44,0.230000,True
5,2019-01,'s-Hertogenbosch,Eindhoven Centraal,257,81,1,1,138,0.316406,True
...,...,...,...,...,...,...,...,...,...,...
51872,2024-12,Zwolle,Roosendaal,1118,320,116,24,1063,0.292505,True
51873,2024-12,Zwolle,Rotterdam Centraal,13,2,0,1,4,0.166667,False
51874,2024-12,Zwolle,Schiphol Airport,24,9,0,0,12,0.375000,True
51875,2024-12,Zwolle,Utrecht Centraal,1163,246,78,14,875,0.214099,True
